In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import gaussian_kde
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

### Data

In [3]:
# df = pd.read_csv('data/pre-thin-data.csv')

# df = pd.read_csv('data/clean-thinning-data-2.csv')

df = pd.read_csv('data/clean-thinning-data-3.csv')

df['status'] = df['pre_HT'].apply(lambda x: 'Alive' if pd.notnull(x) and x > 0 else 'Dead')

### Thinning Strategies

In [4]:
def k_row_thinning(df, k:int, start_row:int=1, row_col='Row', status_col='status'):
    assert k >= 2 and 1 <= start_row <= k
    d = df.copy()
    d['thin_decision'] = 'Dead (ignored)'
    alive = d[status_col].eq('Alive')
    rows_to_thin = ((d[row_col] - start_row) % k == 0)
    d.loc[alive & rows_to_thin, 'thin_decision'] = 'Thin'
    d.loc[alive & ~rows_to_thin, 'thin_decision'] = 'Keep'
    return d

three_row_thinning = lambda df, start_row=1: k_row_thinning(df, 3, start_row)
four_row_thinning  = lambda df, start_row=1: k_row_thinning(df, 4, start_row)
five_row_thinning  = lambda df, start_row=1: k_row_thinning(df, 5, start_row)

def stand_analysis(df_thinned, metric='pre_DBH', vol_col='pre_stem_vol', status_col='status'):
    alive   = df_thinned[df_thinned[status_col] == 'Alive'].copy()
    kept    = alive[alive['thin_decision'] == 'Keep']
    removed = alive[alive['thin_decision'] == 'Thin']

    # volumes
    pre_total_vol     = float(alive[vol_col].sum())   if len(alive) else 0.0
    post_total_vol    = float(kept[vol_col].sum())    if len(kept)  else 0.0
    removed_total_vol = float(removed[vol_col].sum()) if len(removed) else 0.0

    # central tendencies
    pre_median  = float(alive[metric].median()) if len(alive) else np.nan
    pre_mean    = float(alive[metric].mean())   if len(alive) else np.nan
    post_median = float(kept[metric].median())  if len(kept)  else np.nan
    post_mean   = float(kept[metric].mean())    if len(kept)  else np.nan
    d_median    = (post_median - pre_median) if (not np.isnan(post_median) and not np.isnan(pre_median)) else np.nan
    d_mean      = (post_mean   - pre_mean)   if (not np.isnan(post_mean)   and not np.isnan(pre_mean))   else np.nan

    # quartiles
    q1 = alive[metric].quantile(0.25) if len(alive) else np.nan
    q3 = alive[metric].quantile(0.75) if len(alive) else np.nan

    # counts by quartile
    L_pre  = int((alive [metric] <= q1).sum()) if len(alive) else 0  # Q1
    H_pre  = int((alive [metric] >= q3).sum()) if len(alive) else 0  # Q4
    L_post = int((kept  [metric] <= q1).sum()) if len(kept)  else 0
    H_post = int((kept  [metric] >= q3).sum()) if len(kept)  else 0

    # removed by quartile
    L_cut = L_pre - L_post
    H_cut = H_pre - H_post

    # volume by Q4 removed
    removed_q4_vol = float(removed.loc[removed[metric] >= q3, vol_col].sum()) if len(removed) else 0.0
    pct_removed_vol_from_q4 = (100 * removed_q4_vol / removed_total_vol) if removed_total_vol > 0 else np.nan

    # key ratios
    r_q4 = (H_post / H_pre) if H_pre > 0 else np.nan              
    r_q1 = (L_cut / L_pre) if L_pre > 0 else np.nan               

    return {
        # overview
        'Trees removed(%)': round((len(removed) / len(alive)) * 100, 2) if len(alive) else np.nan,
        'Volume removed(%)': round((removed_total_vol / pre_total_vol) * 100, 2) if pre_total_vol > 0 else np.nan,
        'Trees kept': int(len(kept)),
        'Trees removed': int(len(removed)),

        # quartile event columns
        'Q1 cut (count)': int(L_cut),
        'Q4 cut (count)': int(H_cut),
        'Q4 remaining (count)': int(H_post),
        'Removed volume-Q4': round(removed_q4_vol, 2),
        'Removed volume-Q4(%)': round(pct_removed_vol_from_q4, 2) if not np.isnan(pct_removed_vol_from_q4) else np.nan,

        # ratios
        'Q1 removal ratio': round(r_q1, 3) if not np.isnan(r_q1) else np.nan,
        'Q4 retention ratio': round(r_q4, 3) if not np.isnan(r_q4) else np.nan,

        'Q1 pre (count)': int(L_pre),
        'Q4 pre (count)': int(H_pre),

        # DBH/volume stats
        'Post-thinning total volume': round(post_total_vol, 2),
        'Volume removed': round(removed_total_vol, 2),
        'Pre-thinning Median DBH': round(pre_median, 2) if not np.isnan(pre_median) else np.nan,
        'Pre-thinning Mean DBH': round(pre_mean, 2)     if not np.isnan(pre_mean)   else np.nan,
        'Post-thinning Median DBH': round(post_median, 2) if not np.isnan(post_median) else np.nan,
        'Post-thinning Mean DBH': round(post_mean, 2)     if not np.isnan(post_mean)   else np.nan,
        'Change-Median DBH': round(d_median, 2) if not np.isnan(d_median) else np.nan,
        'Change-Mean DBH':   round(d_mean, 2)   if not np.isnan(d_mean)   else np.nan,
    }

def build_comp_for_k(df, k:int, metric='pre_DBH', vol_col='pre_stem_vol',
                     row_col='Row', status_col='status'):
    rows = []
    for s in range(1, k+1):
        d = k_row_thinning(df, k, s, row_col=row_col, status_col=status_col)
        a = stand_analysis(d, metric=metric, vol_col=vol_col, status_col=status_col)
        a['Strategy']  = f'{k}-row start={s}'
        a['k']         = k
        a['start_row'] = s
        rows.append(a)
    comp = pd.DataFrame(rows)
    for c in ('Q4 retention ratio', 'Q1 removal ratio'):
        if c in comp.columns: comp[c] = pd.to_numeric(comp[c], errors='coerce')
    return comp

from functools import lru_cache

def variable_row_thinning(df, cut_rows, row_col='Row', status_col='status'):
    d = df.copy()
    d['thin_decision'] = 'Dead (ignored)'
    alive = d[status_col].eq('Alive')
    in_cut = d[row_col].isin(cut_rows)
    d.loc[alive & in_cut,  'thin_decision'] = 'Thin'
    d.loc[alive & ~in_cut, 'thin_decision'] = 'Keep'
    return d

def _row_q4_volume_by_row(df, metric='pre_DBH', vol_col='pre_stem_vol',
                          row_col='Row', status_col='status'):
    alive = df[df[status_col] == 'Alive'].copy()
    if alive.empty:
        raise ValueError("No Alive trees found; cannot compute Q4 volumes.")
    q3 = float(alive[metric].quantile(0.75))
    rows = np.sort(pd.unique(df[row_col]))
    q4_rows = alive.loc[alive[metric] >= q3, [row_col, vol_col]]
    q4_vol_by_row = q4_rows.groupby(row_col)[vol_col].sum()
    q4_vol_by_row = q4_vol_by_row.reindex(rows, fill_value=0.0)
    return rows, q3, q4_vol_by_row.values

def _best_sequence_from_start_q4vol(rows, q4_vols, start_idx, target_cuts, steps=(3,4,5)):

    N = len(rows)
    if target_cuts <= 0 or start_idx < 0 or start_idx >= N:
        return None
    if start_idx + 3*(target_cuts-1) > N-1:
        return None

    @lru_cache(maxsize=None)
    def dp(last_idx, selected):
        if selected == target_cuts:
            return (0.0, ())
        remaining = target_cuts - selected
        if last_idx + 3*remaining > N-1:
            return None
        best = None
        cand = []
        for st in steps:
            nxt = last_idx + st
            if nxt <= N-1:
                cand.append((q4_vols[nxt], st, nxt))
        cand.sort(key=lambda x: (x[0], x[1], x[2]))
        for q4v, st, nxt in cand:
            rem_after = target_cuts - (selected + 1)
            if nxt + 3*rem_after > N-1:
                continue
            sub = dp(nxt, selected + 1)
            if sub is None:
                continue
            sub_q4, sub_path = sub
            cand_val = (q4v + sub_q4, (nxt,) + sub_path)
            if best is None or (cand_val[0] < best[0]) or (cand_val[0] == best[0] and cand_val[1] < best[1]):
                best = cand_val
        return best

    start_cost = float(q4_vols[start_idx])
    sub = dp(start_idx, 1)
    if sub is None:
        return None
    sub_q4, sub_path = sub
    total_q4 = start_cost + sub_q4
    path = (start_idx,) + sub_path
    return (total_q4, path)

def choose_variable_cut_rows_q4volume(df, target_cuts:int, metric='pre_DBH', vol_col='pre_stem_vol',
                                      row_col='Row', status_col='status',
                                      first_start_rows:int=5,
                                      min_in_between:int=2, max_in_between:int=4):
    assert min_in_between == 2 and max_in_between == 4, "This version fixes steps to {3,4,5}."
    rows, q3, q4_vols = _row_q4_volume_by_row(df, metric=metric, vol_col=vol_col,
                                              row_col=row_col, status_col=status_col)
    N = len(rows)
    if N == 0:
        return []

    max_start_idx = min(first_start_rows, N) - 1
    feasible_starts = [s for s in range(0, max_start_idx + 1) if s + 3*(target_cuts-1) <= N-1]
    if not feasible_starts:
        raise ValueError(f"Infeasible: cannot place {target_cuts} cuts starting within first {first_start_rows} rows.")

    best_total = None
    best_path  = None
    best_start = None
    for s in feasible_starts:
        res = _best_sequence_from_start_q4vol(rows, q4_vols, s, target_cuts, steps=(3,4,5))
        if res is None:
            continue
        tot_q4, path = res
        if (best_total is None) or (tot_q4 < best_total) or (tot_q4 == best_total and s < best_start):
            best_total = tot_q4
            best_path  = path
            best_start = s

    if best_path is None:
        raise ValueError("No feasible sequence found.")
    return rows[list(best_path)].tolist()

def choose_variable_cut_rows_q4volume_gap34(
    df, target_cuts:int, metric='pre_DBH', vol_col='pre_stem_vol',
    row_col='Row', status_col='status',
    first_start_rows:int=5,
    min_in_between:int=3, max_in_between:int=4
):

    assert min_in_between == 3 and max_in_between == 4, "This variant fixes steps to {4,5}."
    rows, q3, q4_vols = _row_q4_volume_by_row(
        df, metric=metric, vol_col=vol_col, row_col=row_col, status_col=status_col
    )
    N = len(rows)
    if N == 0:
        return []

    min_step = 4  
    max_start_idx = min(first_start_rows, N) - 1
    feasible_starts = [s for s in range(0, max_start_idx + 1)
                       if s + min_step * (target_cuts - 1) <= N - 1]
    if not feasible_starts:
        raise ValueError(
            f"Infeasible: cannot place {target_cuts} cuts with gaps [3,4] "
            f"starting within first {first_start_rows} rows."
        )

    best_total = None
    best_path  = None
    best_start = None
    for s in feasible_starts:
        
        res = _best_sequence_from_start_q4vol(rows, q4_vols, s, target_cuts, steps=(4, 5))
        if res is None:
            continue
        tot_q4, path = res
        if (best_total is None) or (tot_q4 < best_total) or (tot_q4 == best_total and s < best_start):
            best_total = tot_q4
            best_path  = path
            best_start = s

    if best_path is None:
        raise ValueError("No feasible sequence found.")
    return rows[list(best_path)].tolist()

def variable_thinning_variants_volume_pure(
    df, metric='pre_DBH', vol_col='pre_stem_vol',
    row_col='Row', status_col='status'
):

    rows_sorted = np.sort(pd.unique(df[row_col]))
    R = len(rows_sorted)
    targets = {
        '3_row_eqv': R // 3,
        '4_row_eqv': R // 4,
        '5_row_eqv': R // 5,
    }

    out = {}
    for label, m in targets.items():
        if label == '5_row_eqv':
            
            cuts = choose_variable_cut_rows_q4volume_gap34(
                df, m, metric=metric, vol_col=vol_col, row_col=row_col, status_col=status_col,
                first_start_rows=5, min_in_between=3, max_in_between=4
            )
        else:
            
            cuts = choose_variable_cut_rows_q4volume(
                df, m, metric=metric, vol_col=vol_col, row_col=row_col, status_col=status_col,
                first_start_rows=5, min_in_between=2, max_in_between=4
            )
        d_thin = variable_row_thinning(df, cuts, row_col=row_col, status_col=status_col)
        out[label] = (cuts, d_thin)
    return out


def build_comp_for_variable_variants(variants_dict, metric='pre_DBH', vol_col='pre_stem_vol', status_col='status'):

    rows = []
    for label, (cut_rows, d_thin) in variants_dict.items():
        a = stand_analysis(d_thin, metric=metric, vol_col=vol_col, status_col=status_col)
        a['Strategy']  = label
        a['k']         = 'variable'
        a['start_row'] = cut_rows[0] if len(cut_rows) else np.nan
        rows.append(a)
    comp = pd.DataFrame(rows)
    for c in ('Q4 retention ratio', 'Q1 removal ratio'):
        if c in comp.columns: comp[c] = pd.to_numeric(comp[c], errors='coerce')
    return comp

def adaptive_practical_band(q4_series: pd.Series, lam: float = 0.5,
                            delta_min: float = 0.002, delta_max: float = 0.012) -> float:
    q4 = pd.to_numeric(q4_series, errors='coerce')
    rng = float(q4.max() - q4.min()) if q4.notna().any() else 0.0
    return float(np.clip(lam * rng, delta_min, delta_max))

def rank_by_lexi_apb_strict(comp: pd.DataFrame,
                            p_col: str = 'Q4 retention ratio',
                            s_col: str = 'Q1 removal ratio',
                            lam: float = 0.5, delta_min: float = 0.002, delta_max: float = 0.012,
                            delta_override: float | None = None) -> pd.DataFrame:

    df = comp.copy()
    p = pd.to_numeric(df[p_col], errors='coerce').fillna(-np.inf)
    s = pd.to_numeric(df[s_col], errors='coerce').fillna(-np.inf)

    delta  = float(delta_override) if delta_override is not None else adaptive_practical_band(p, lam, delta_min, delta_max)
    p_best = float(p.max())
    close  = (p_best - p) <= delta
    df['_apb_close'] = close

    df['_apb_s_key'] = np.where(close, s, -np.inf)
    df['_apb_p_key'] = p

    def _norm(x):
        xmin, xmax = float(x.min()), float(x.max())
        return (x - xmin) / (xmax - xmin) if xmax > xmin else pd.Series(0.5, index=x.index)
    df['Lexi-APB index'] = close.astype(float) + np.where(close, _norm(s), _norm(p)) * 1e-3

    df = (df.sort_values(by=['_apb_close', '_apb_s_key', '_apb_p_key'],
                         ascending=[False,        False,       False])
            .drop(columns=['_apb_close','_apb_s_key','_apb_p_key'])
            .reset_index(drop=True))

    df.attrs['apb_delta'] = float(delta)
    return df

def style_comp_table(comp_ranked: pd.DataFrame, title=None):
    show = [
        'Strategy',
        'Lexi-APB index',
        'Trees removed(%)','Volume removed(%)',
        'Trees kept','Trees removed',
        'Q1 cut (count)','Q1 removal ratio',
        'Q4 remaining (count)','Q4 cut (count)',
        'Q4 retention ratio',
        'Removed volume-Q4(%)','Removed volume-Q4',
        'Volume removed','Post-thinning total volume',
        'start_row'
    ]
    cols = [c for c in show if c in comp_ranked.columns]
    tbl  = comp_ranked[cols].copy()

    count_cols = ['Trees kept','Trees removed','Q1 cut (count)','Q4 cut (count)','Q4 remaining (count)']
    pct_cols   = ['Trees removed(%)','Volume removed(%)','Removed volume-Q4(%)']
    ratio_cols = ['Q4 retention ratio','Q1 removal ratio']
    money_cols = ['Removed volume-Q4','Volume removed','Post-thinning total volume']

    sty = (tbl.reset_index(drop=True)
           .style
           .format({c:'{:,.0f}' for c in count_cols if c in tbl})
           .format({c:'{:.2f}%' for c in pct_cols   if c in tbl})
           .format({c:'{:.3f}'  for c in ratio_cols if c in tbl})
           .format({c:'{:,.6f}' for c in ['Lexi-APB index'] if c in tbl})
           .format({c:'{:,.2f}' for c in money_cols if c in tbl})
           .set_caption(title or 'Strategies ranked — Lexi-APB')
           .set_properties(**{'font-variant-numeric':'tabular-nums'})
           .hide(axis='index'))
    return sty

def score_krow_lexi_apb(df, k:int, metric='pre_DBH', vol_col='pre_stem_vol',
                        row_col='Row', status_col='status',
                        lam:float=0.5, delta_min:float=0.002, delta_max:float=0.012,
                        delta_override:float=None, title=None, return_comp=False):
    comp = build_comp_for_k(df, k, metric=metric, vol_col=vol_col, row_col=row_col, status_col=status_col)
    ranked = rank_by_lexi_apb_strict(comp,
                                     p_col='Q4 retention ratio',
                                     s_col='Q1 removal ratio',
                                     lam=lam, delta_min=delta_min, delta_max=delta_max,
                                     delta_override=delta_override)
    apb = ranked.attrs.get('apb_delta', None)
    title = title or f'{k}-row — Lexi-APB (δ={apb:.4f})'
    sty = style_comp_table(ranked, title)
    if return_comp:
        return sty, ranked
    return sty

def score_variable_variants_lexi_apb(df, metric='pre_DBH', vol_col='pre_stem_vol',
                                     row_col='Row', status_col='status',
                                     lam:float=0.5, delta_min:float=0.002, delta_max:float=0.012,
                                     delta_override:float=None, title=None, return_comp=False):
    variants = variable_thinning_variants_volume_pure(df, metric=metric, vol_col=vol_col,
                                                      row_col=row_col, status_col=status_col)
    comp = build_comp_for_variable_variants(variants, metric=metric, vol_col=vol_col, status_col=status_col)
    ranked = rank_by_lexi_apb_strict(comp,
                                     p_col='Q4 retention ratio',
                                     s_col='Q1 removal ratio',
                                     lam=lam, delta_min=delta_min, delta_max=delta_max,
                                     delta_override=delta_override)
    apb = ranked.attrs.get('apb_delta', None)
    title = title or f'Variable thinning — Lexi-APB (δ={apb:.4f})'
    sty = style_comp_table(ranked, title)
    if return_comp:
        return sty, ranked, variants
    return sty

# sty5, ranked5 = score_krow_lexi_apb(df, k=5, return_comp=True)
# display(sty5); print('Winner (5-row): start_row =', int(ranked5.iloc[0]['start_row']))

# sty4, ranked4 = score_krow_lexi_apb(df, k=4, return_comp=True)
# display(sty4); print('Winner (4-row): start_row =', int(ranked4.iloc[0]['start_row']))

# sty3, ranked3 = score_krow_lexi_apb(df, k=3, return_comp=True)
# display(sty3); print('Winner (3-row): start_row =', int(ranked3.iloc[0]['start_row']))

# sty_var, ranked_var, variants = score_variable_variants_lexi_apb(df, return_comp=True)
# display(sty_var); print('Winner (variable):', ranked_var.iloc[0]['Strategy'])


### User Interface

In [5]:
def plot_thinning_map(df_thinned, row_col='Row', x_col='Tree', status_col='status', title='Selected strategy — spatial map'):
    alive = df_thinned[df_thinned[status_col] == 'Alive']
    kept = alive[alive['thin_decision'] == 'Keep']
    thinned = alive[alive['thin_decision'] == 'Thin']
    dead = df_thinned[df_thinned[status_col] == 'Dead']

    fig, ax = plt.subplots(figsize=(8, 8))

    if not dead.empty:
        ax.scatter(dead[x_col], dead[row_col], s=18, c='gray', alpha=0.5, label=f'Dead (n={len(dead)})', edgecolors='none')
    if not kept.empty:
        ax.scatter(kept[x_col], kept[row_col], s=21, c='green', label=f'Kept (n={len(kept)})', edgecolors='k', linewidths=0.25)
    if not thinned.empty:
        ax.scatter(thinned[x_col], thinned[row_col], s=18, c='red', alpha=0.3 , label=f'Thinned (n={len(thinned)})')

    ax.invert_yaxis()
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlabel('Tree (column)')
    ax.set_ylabel('Row')
    ax.set_title(title)
    ax.grid(True, linewidth=0.5, alpha=0.5)
    ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1), borderaxespad=0.)
    fig.subplots_adjust(right=0.78)
    plt.tight_layout()
    plt.show()

def style_comp_table_detailed(comp_ranked: pd.DataFrame, title=None):
    show = [
        'Strategy',
        'Trees removed(%)','Volume removed(%)',
        'Trees kept','Trees removed',
        'Q1 cut (count)',
        'Q4 remaining (count)','Q4 cut (count)',
        'Removed volume-Q4(%)','Removed volume-Q4',
        'Change-Median DBH','Change-Mean DBH',
        'Volume removed','Post-thinning total volume',
        'Q1 removal ratio',
        'Q4 retention ratio',

        'Pre-thinning Median DBH','Pre-thinning Mean DBH',
        'Post-thinning Median DBH','Post-thinning Mean DBH',
        'Lexi-APB index'

        
    ]
    cols = [c for c in show if c in comp_ranked.columns]
    tbl  = comp_ranked[cols].copy()

    count_cols = ['Trees kept','Trees removed','Q1 cut (count)','Q4 cut (count)','Q4 remaining (count)']
    pct_cols   = ['Trees removed(%)','Volume removed(%)','Removed volume-Q4(%)']
    ratio_cols = ['Q4 retention ratio','Q1 removal ratio']
    money_cols = ['Removed volume-Q4','Volume removed','Post-thinning total volume']
    dbh_cols   = ['Pre-thinning Mean DBH','Post-thinning Mean DBH',
                  'Pre-thinning Median DBH','Post-thinning Median DBH',
                  'Change in Median DBH','Change in Mean DBH']

    sty = (tbl.reset_index(drop=True)
           .style
           .format({c:'{:,.0f}' for c in count_cols if c in tbl})
           .format({c:'{:.2f}%' for c in pct_cols   if c in tbl})
           .format({c:'{:.3f}'  for c in ratio_cols if c in tbl})
           .format({c:'{:,.6f}' for c in ['Lexi-APB index'] if c in tbl})
           .format({c:'{:,.2f}' for c in money_cols if c in tbl})
           .format({c:'{:,.2f}' for c in dbh_cols   if c in tbl})
           .set_caption(title or 'Strategies ranked — Lexi-APB')
           .set_properties(**{'font-variant-numeric':'tabular-nums'})
           .hide(axis='index'))
    return sty


def _best_krow_result(df, k:int,
                      metric='pre_DBH', vol_col='pre_stem_vol',
                      row_col='Row', status_col='status',
                      lam=0.5, delta_min=0.002, delta_max=0.012, delta_override=None):

    comp = build_comp_for_k(df, k, metric=metric, vol_col=vol_col, row_col=row_col, status_col=status_col)
    ranked = rank_by_lexi_apb_strict(comp,
                                     p_col='Q4 retention ratio',
                                     s_col='Q1 removal ratio',
                                     lam=lam, delta_min=delta_min, delta_max=delta_max,
                                     delta_override=delta_override)
    apb = ranked.attrs.get('apb_delta', np.nan)
    sty = style_comp_table_detailed(ranked, title=f'{k}-row — Lexi-APB (δ={apb:.4f})')

    # best start
    best_start = int(ranked.iloc[0]['start_row'])
    df_best = k_row_thinning(df, k, start_row=best_start, row_col=row_col, status_col=status_col)
    return sty, ranked, df_best

def _variable_equivalent_for_k(df, k:int,
                               metric='pre_DBH', vol_col='pre_stem_vol',
                               row_col='Row', status_col='status'):

    variants = variable_thinning_variants_volume_pure(df, metric=metric, vol_col=vol_col,
                                                      row_col=row_col, status_col=status_col)
    key = {3: '3_row_eqv', 4: '4_row_eqv', 5: '5_row_eqv'}[k]
    cut_rows, d_thin = variants[key]

    a = stand_analysis(d_thin, metric=metric, vol_col=vol_col, status_col=status_col)
    a['Strategy']  = f'variable-{key}'
    a['k']         = 'variable'
    a['start_row'] = cut_rows[0] if len(cut_rows) else np.nan
    comp_var = pd.DataFrame([a])

    comp_var['Lexi-APB index'] = 1.0  # cosmetic

    sty_var = style_comp_table_detailed(comp_var, title=f'Variable equivalent — {key}')
    return sty_var, comp_var, d_thin

ddl = widgets.Dropdown(
    options=[('3-row thinning','3'), ('4-row thinning','4'), ('5-row thinning','5')],
    value='3',
    description='Thinning:',
    layout=widgets.Layout(width='240px')
)

table_k_out   = widgets.Output()
table_var_out = widgets.Output()
map_best_out  = widgets.Output()
map_var_out   = widgets.Output()
note_out      = widgets.Output()

def render():
    with table_k_out:   clear_output(wait=True)
    with table_var_out: clear_output(wait=True)
    with map_best_out:  clear_output(wait=True)
    with map_var_out:   clear_output(wait=True)
    with note_out:      clear_output(wait=True)

    k = int(ddl.value)


    sty_k, ranked_k, df_best = _best_krow_result(df, k=k)
    with table_k_out:
        display(sty_k)

    sty_var, comp_var, df_var = _variable_equivalent_for_k(df, k=k)
    with table_var_out:
        display(sty_var)

    with note_out:
        best_start = int(ranked_k.iloc[0]['start_row'])
        print(f'Winner ({k}-row): start_row = {best_start}')
        

    with map_best_out:
        plot_thinning_map(df_best, row_col='Row', x_col='Tree', status_col='status',
                          title=f'Spatial map — Best {k}-row strategy')
    with map_var_out:
        plot_thinning_map(df_var, row_col='Row', x_col='Tree', status_col='status',
                          title=f'Spatial map — Variable equivalent for {k}-row')

ddl.observe(lambda ch: render(), names='value')

header = widgets.HBox([
    widgets.HTML("<b>Select thinning strategy:</b>"),
    ddl
], layout=widgets.Layout(align_items='center', gap='12px'))

tables = widgets.VBox([
    table_k_out,
    widgets.HTML("<hr style='margin:8px 0;'>"),
    table_var_out
])

maps = widgets.VBox([
    note_out,
    map_best_out,
    map_var_out
])

ui = widgets.VBox([
    header,
    widgets.HTML("<hr>"),
    tables,
    widgets.HTML("<hr>"),
    maps
], layout=widgets.Layout(width='100%'))

display(ui)
render()


#### Separating the best 3-row thinned stand

In [14]:
sty3, ranked3 = score_krow_lexi_apb(df, k=3, return_comp=True)

best_start3 = int(ranked3.loc[0, 'start_row'])

df_best3 = k_row_thinning(df, k=3, start_row=best_start3, row_col='Row', status_col='status')


### Thin from Below

In [7]:
def thin_from_below_adjacent_simple(df_best3,
                                    fraction: float = 1/3,
                                    metric: str = 'pre_DBH',
                                    row_col: str = 'Row',
                                    status_col: str = 'status'):

    d = df_best3.copy()

    alive = d[status_col].eq('Alive')
    corridor_mask = alive & d['thin_decision'].eq('Thin')
    corridor_rows = np.sort(d.loc[corridor_mask, row_col].unique())

    if len(corridor_rows) == 0:
        return d, {'corridor_rows': [], 'side_rows': [], 'n_removed_below': 0}

    all_rows = np.sort(d[row_col].unique())
    all_rows_set = set(all_rows)

    side_rows = set()
    for rc in corridor_rows:
        if (rc - 1) in all_rows_set:
            side_rows.add(rc - 1)
        if (rc + 1) in all_rows_set:
            side_rows.add(rc + 1)
    side_rows = sorted([r for r in side_rows if r not in set(corridor_rows)])

    to_remove_idx = []
    for r in side_rows:
        elig = alive & d['thin_decision'].eq('Keep') & d[row_col].eq(r)
        sub = d.loc[elig, [metric]]
        n = len(sub)
        if n == 0:
            continue
        k_rm = int(np.floor(n * fraction))
        if k_rm <= 0:
            continue

        idx = sub.nsmallest(k_rm, metric).index
        to_remove_idx.append(idx)

    if len(to_remove_idx) > 0:
        to_remove_idx = pd.Index(np.concatenate([ix.values for ix in to_remove_idx]))
        d.loc[to_remove_idx, 'thin_decision'] = 'Thin'
        n_removed_below = int(len(to_remove_idx))
    else:
        n_removed_below = 0

    info = {
        'corridor_rows': corridor_rows.tolist(),
        'side_rows': side_rows,
        'n_removed_below': n_removed_below
    }
    return d, info

df_tfb, tfb_info = thin_from_below_adjacent_simple(
    df_best3,
    fraction=1/3,          
    metric='pre_DBH',
    row_col='Row',
    status_col='status'
)

def _stand_metrics_relative(base_df,final_df,base_mask,strategy,
                            metric='pre_DBH',vol_col='pre_stem_vol',
                            status_col='status',thin_col='thin_decision',
                            keep_val='Keep',thin_val='Thin'):

    base_idx=base_df.index[base_mask]
    base_idx=base_idx.intersection(final_df.index)

    if len(base_idx)==0:
        return {
            'Strategy':strategy,
            'Trees removed(%)':0.0,'Volume removed(%)':0.0,
            'Trees kept':0,'Trees removed':0,
            'Q1 cut (count)':0,
            'Q4 remaining (count)':0,'Q4 cut (count)':0,
            'Removed volume-Q4(%)':0.0,'Removed volume-Q4':0.0,
            'Change-Median DBH':np.nan,'Change-Mean DBH':np.nan,
            'Volume removed':0.0,'Post-thinning total volume':0.0,
            'Q1 removal ratio':np.nan,'Q4 retention ratio':np.nan,
            'Pre-thinning Median DBH':np.nan,'Pre-thinning Mean DBH':np.nan,
            'Post-thinning Median DBH':np.nan,'Post-thinning Mean DBH':np.nan
        }

    pre=base_df.loc[base_idx]
    post=final_df.loc[base_idx]

    keep_mask=post[thin_col].eq(keep_val)
    cut_mask=post[thin_col].eq(thin_val)

    n_base=len(base_idx)
    n_kept=int(keep_mask.sum())
    n_cut=int(cut_mask.sum())


    base_vol=float(pre[vol_col].sum())
    vol_cut=float(pre.loc[cut_mask,vol_col].sum())
    vol_post=float(pre.loc[keep_mask,vol_col].sum())

    trees_removed_pct=100*(n_cut/n_base) if n_base else 0.0
    vol_removed_pct=100*(vol_cut/base_vol) if base_vol>0 else 0.0

    x=pre[metric].astype(float)
    q1_thr=float(x.quantile(0.25))
    q3_thr=float(x.quantile(0.75))
    q1_mask=x<=q1_thr
    q4_mask=x>=q3_thr

    q1_cut=int((q1_mask&cut_mask).sum())
    q4_keep=int((q4_mask&keep_mask).sum())
    q4_cut=int((q4_mask&cut_mask).sum())

    vol_cut_q4=float(pre.loc[q4_mask&cut_mask,vol_col].sum())
    vol_cut_q4_pct=100*(vol_cut_q4/vol_cut) if vol_cut>0 else 0.0

    pre_med=float(x.median()); pre_mean=float(x.mean())
    x_post=pre.loc[keep_mask,metric].astype(float)
    post_med=float(x_post.median()) if len(x_post)>0 else np.nan
    post_mean=float(x_post.mean()) if len(x_post)>0 else np.nan

    change_med=post_med-pre_med if pd.notnull(post_med) else np.nan
    change_mean=post_mean-pre_mean if pd.notnull(post_mean) else np.nan

    base_q1=int(q1_mask.sum()); base_q4=int(q4_mask.sum())
    q1_removal_ratio=(q1_cut/base_q1) if base_q1>0 else np.nan
    q4_retention_ratio=(q4_keep/base_q4) if base_q4>0 else np.nan

    return {
        'Strategy':strategy,
        'Trees removed(%)':trees_removed_pct,
        'Volume removed(%)':vol_removed_pct,
        'Trees kept':n_kept,
        'Trees removed':n_cut,
        'Q1 cut (count)':q1_cut,
        'Q4 remaining (count)':q4_keep,
        'Q4 cut (count)':q4_cut,
        'Removed volume-Q4(%)':vol_cut_q4_pct,   
        'Removed volume-Q4':vol_cut_q4,
        'Change-Median DBH':change_med,
        'Change-Mean DBH':change_mean,
        'Volume removed':vol_cut,
        'Post-thinning total volume':vol_post,
        'Q1 removal ratio':q1_removal_ratio,    
        'Q4 retention ratio':q4_retention_ratio, 
        'Pre-thinning Median DBH':pre_med,
        'Pre-thinning Mean DBH':pre_mean,
        'Post-thinning Median DBH':post_med,
        'Post-thinning Mean DBH':post_mean
    }

def table_final_vs_initial(df_initial,df_final,*,metric='pre_DBH',vol_col='pre_stem_vol',
                           status_col='status',thin_col='thin_decision',
                           keep_val='Keep',thin_val='Thin',strategy='Final vs Initial (Alive baseline)'):
    base_mask=df_initial[status_col].eq('Alive')
    rep=_stand_metrics_relative(df_initial,df_final,base_mask,strategy,metric,vol_col,status_col,thin_col,keep_val,thin_val)
    return pd.DataFrame([rep])

def table_final_vs_after_first(df_after_first,df_final,*,metric='pre_DBH',vol_col='pre_stem_vol',
                               status_col='status',thin_col='thin_decision',
                               keep_val='Keep',thin_val='Thin',strategy='Final vs After 1st Thinning (Alive&Keep baseline)'):
    base_mask=df_after_first[status_col].eq('Alive')&df_after_first[thin_col].eq(keep_val)
    rep=_stand_metrics_relative(df_after_first,df_final,base_mask,strategy,metric,vol_col,status_col,thin_col,keep_val,thin_val)
    return pd.DataFrame([rep])


tbl_vs_initial = table_final_vs_initial(
    df_initial=df,
    df_final=df_tfb,
    metric='pre_DBH',
    vol_col='pre_stem_vol',
    status_col='status',
    thin_col='thin_decision',
    strategy='Thin from below vs Initial'
).round(3).set_index('Strategy')

tbl_vs_after_first = table_final_vs_after_first(
    df_after_first=df_best3,
    df_final=df_tfb,
    metric='pre_DBH',
    vol_col='pre_stem_vol',
    status_col='status',
    thin_col='thin_decision',
    strategy='Thin from below vs After 3-row'
).round(3).set_index('Strategy')

# display(tbl_vs_initial)
# display(tbl_vs_after_first)

# plot_thinning_map(df_tfb, row_col='Row', x_col='Tree', status_col='status',
#                   title='Thin feom Below - Spatial Map')

### Thin from Above-1

In [8]:
def thin_from_above_neighbors(df_best3,
                              removal_fraction: float = 1/3,  
                              metric: str = 'pre_DBH',         
                              row_col: str = 'Row',
                              x_col: str = 'Tree',             
                              status_col: str = 'status',
                              thin_col: str = 'thin_decision',
                              keep_val: str = 'Keep',
                              thin_val: str = 'Thin',
                              anchor_fraction: float = 0.10,   
                              min_anchors: int = 1,
                              radius: int | None = None,       
                              combine: str = 'max'             
                              ):


    assert 0 < removal_fraction < 1 
    assert 0 < anchor_fraction <= 1

    d0 = df_best3.copy()
    alive = d0[status_col].eq('Alive')

    corridor_mask = alive & d0[thin_col].eq(thin_val)
    corridor_rows = np.sort(d0.loc[corridor_mask, row_col].unique())
    if len(corridor_rows) == 0:
        return d0, {'corridor_rows': [], 'side_rows': [], 'per_row_quota': {}, 'per_row_removed': {}, 'anchors_per_row': {}}

    all_rows = np.sort(d0[row_col].unique())
    corridor_set = set(corridor_rows)

    corr_to_side = {}
    for rc in corridor_rows:
        side = []
        if (rc - 1) in all_rows and (rc - 1) not in corridor_set: side.append(rc - 1)
        if (rc + 1) in all_rows and (rc + 1) not in corridor_set: side.append(rc + 1)
        corr_to_side[rc] = side

    side_rows_all = sorted({r for rows in corr_to_side.values() for r in rows})
    per_row_quota = {}
    per_row_eligible_idx = {}
    per_row_anchors = {}

    for r in side_rows_all:
        elig = alive & d0[thin_col].eq(keep_val) & d0[row_col].eq(r)
        idx = d0.index[elig]
        n = len(idx)
        if n == 0:
            per_row_quota[r] = 0
            per_row_eligible_idx[r] = idx
            per_row_anchors[r] = pd.Index([])
            continue
        quota = int(np.floor(n * removal_fraction))
        quota = max(0, min(quota, n))
        per_row_quota[r] = quota

        # Anchors: top DBH on this side row
        sub = d0.loc[idx, [metric, x_col]]
        k_anchor = max(min_anchors, int(np.ceil(n * anchor_fraction)))
        k_anchor = min(k_anchor, n)
        anchors_idx = sub.nlargest(k_anchor, metric).index
        per_row_anchors[r] = anchors_idx

        per_row_eligible_idx[r] = idx

    scores = {}
    for rc, side_rows in corr_to_side.items():
        for r in side_rows:
            idx = per_row_eligible_idx[r]
            if len(idx) == 0: 
                continue

            sub = d0.loc[idx, [metric, x_col]]
            anchors_idx = per_row_anchors[r]
            if len(anchors_idx) == 0:
                continue

            anchors = d0.loc[anchors_idx, [metric, x_col]]

            for i, row_i in sub.iterrows():
                if i in anchors_idx:
                    continue  
                xi = float(row_i[x_col])
  
                inf_vals = []
                for a, row_a in anchors.iterrows():
                    dx = abs(xi - float(row_a[x_col]))
                    if (radius is not None) and (dx > radius):
                        continue
                    inf_vals.append(float(row_a[metric]) / (dx + 1.0))
                if not inf_vals:
                    continue
                s_i = max(inf_vals) if combine == 'max' else sum(inf_vals)

                if i in scores:
                    scores[i] = max(scores[i], s_i) if combine == 'max' else (scores[i] + s_i)
                else:
                    scores[i] = s_i

    selected_idx = []
    per_row_removed = {}
    for r in side_rows_all:
        quota = per_row_quota[r]
        if quota <= 0:
            per_row_removed[r] = 0
            continue

        anchors_idx = set(per_row_anchors[r])

        row_idxs = [i for i in per_row_eligible_idx[r] if (i not in anchors_idx) and (i in scores)]
        if len(row_idxs) == 0:
            per_row_removed[r] = 0
            continue

        row_scores = pd.Series({i: scores[i] for i in row_idxs}).sort_values(ascending=False)
        take = row_scores.index[:quota]
        selected_idx.extend(take.tolist())
        per_row_removed[r] = int(len(take))

    df_out = df_best3.copy()
    if selected_idx:
        df_out.loc[selected_idx, thin_col] = thin_val

    info = {
        'corridor_rows': corridor_rows.tolist(),
        'side_rows': side_rows_all,
        'per_row_quota': per_row_quota,
        'anchors_per_row': {r: list(per_row_anchors[r]) for r in side_rows_all},
        'per_row_removed': per_row_removed,
        'n_removed_total': int(len(selected_idx)),
        'params': {
            'removal_fraction': removal_fraction,
            'anchor_fraction': anchor_fraction,
            'min_anchors': min_anchors,
            'radius': radius,
            'combine': combine,
            'metric': metric,
            'x_col': x_col
        }
    }
    return df_out, info

df_tfa2, tfa2_info = thin_from_above_neighbors(
    df_best3,
    removal_fraction=1/3,     
    metric='pre_DBH',
    row_col='Row',
    x_col='Tree',             
    status_col='status',
    thin_col='thin_decision',
    keep_val='Keep',
    thin_val='Thin',
    anchor_fraction=0.10,     
    min_anchors=1,
    radius=None,              
    combine='max'             
)


def top25_release_table(df_after_first, df_final, *,
                        treatment: str,
                        dbh_col='pre_DBH',
                        row_col='Row',
                        tree_col='Tree',
                        status_col='status',
                        thin_col='thin_decision',
                        keep_val='Keep',
                        thin_val='Thin',
                        neighbors_k: int = 5,
                        row_scale: float = 1.0,
                        tree_scale: float = 1.0) -> pd.DataFrame:

    base_mask = df_after_first[status_col].eq('Alive') & df_after_first[thin_col].eq(keep_val)
    base_idx = df_after_first.index[base_mask]
    if len(base_idx) == 0:
        cols = ['Treatment','Initial tree quantity','Post-thin tree quantity',
                'pre-thin mean DBH','post-thin mean DBH',
                'Top 25% DBH trees (count)',
                'Top 25% DBH trees cut off (count)',
                'No. of Top 25% DBH trees with 5-sided release',
                'No. of Top 25% DBH trees with 4-sided release',
                'No. of Top 25% DBH trees with 3-sided release',
                'No. of Top 25% DBH trees with 2-sided release',
                'No. of Top 25% DBH trees with 1-sided release',
                'No. of Top 25% DBH trees with 0-sided release']
        return pd.DataFrame([dict(zip(cols, [treatment,0,0,np.nan,np.nan,0,0,0,0,0,0,0]))])

    d0 = df_after_first.loc[base_idx]
    d1 = df_final.loc[base_idx]  

    initial_qty = len(base_idx)
    post_keep_mask = d1[thin_col].eq(keep_val)
    post_qty = int(post_keep_mask.sum())

    pre_mean_dbh  = float(d0[dbh_col].astype(float).mean())
    post_mean_dbh = float(d0.loc[post_keep_mask, dbh_col].astype(float).mean()) if post_qty > 0 else np.nan

    x = d0[dbh_col].astype(float)
    q3_thr = float(x.quantile(0.75))
    big_mask = x >= q3_thr
    big_idx  = d0.index[big_mask]
    big_total = int(len(big_idx))

    if big_total == 0:
        out = {
            'Treatment': treatment,
            'Initial tree quantity': initial_qty,
            'Post-thin tree quantity': post_qty,
            'pre-thin mean DBH': pre_mean_dbh,
            'post-thin mean DBH': post_mean_dbh,
            'Top 25% DBH trees (count)': 0,
            'Top 25% DBH trees cut off (count)': 0,
            'No. of Top 25% DBH trees with 5-sided release': 0,
            'No. of Top 25% DBH trees with 4-sided release': 0,
            'No. of Top 25% DBH trees with 3-sided release': 0,
            'No. of Top 25% DBH trees with 2-sided release': 0,
            'No. of Top 25% DBH trees with 1-sided release': 0,
            'No. of Top 25% DBH trees with 0-sided release': 0
        }
        return pd.DataFrame([out])
    big_cut_bool  = d1.loc[big_idx, thin_col].eq(thin_val)
    big_cut_ids   = big_cut_bool[big_cut_bool].index
    big_cut_count = int(len(big_cut_ids))
    big_alive_idx = big_idx.difference(big_cut_ids)

    base_rows  = d0[row_col].to_numpy(float)
    base_trees = d0[tree_col].to_numpy(float)
    base_ids   = np.array(list(base_idx))                  
    id_to_pos  = {base_ids[pos]: pos for pos in range(len(base_ids))}

    buckets = {k: 0 for k in range(0, 6)}

    for bid in big_alive_idx:
        pos = id_to_pos.get(bid, None)
        if pos is None:
            continue

        r0 = float(base_rows[pos]); t0 = float(base_trees[pos])
        dr = (base_rows - r0) * row_scale
        dt = (base_trees - t0) * tree_scale
        dist = np.sqrt(dr*dr + dt*dt)

        dist[pos] = np.inf

        k = min(int(neighbors_k), len(base_idx) - 1)
        if k <= 0:
            buckets[0] += 1
            continue

        nn_pos = np.argpartition(dist, k-1)[:k]
        nn_ids = base_ids[nn_pos]

        nn_cut = int(d1.loc[nn_ids, thin_col].eq(thin_val).sum())
        nn_cut = max(0, min(nn_cut, 5)) 
        buckets[nn_cut] += 1

    out = {
        'Treatment': treatment,
        'Initial tree quantity': initial_qty,
        'Post-thin tree quantity': post_qty,
        'pre-thin mean DBH': pre_mean_dbh,
        'post-thin mean DBH': post_mean_dbh,
        'Top 25% DBH trees (count)': big_total,
        'Top 25% DBH trees cut off (count)': big_cut_count,
        'No. of Top 25% DBH trees with 5-sided release': buckets[5],
        'No. of Top 25% DBH trees with 4-sided release': buckets[4],
        'No. of Top 25% DBH trees with 3-sided release': buckets[3],
        'No. of Top 25% DBH trees with 2-sided release': buckets[2],
        'No. of Top 25% DBH trees with 1-sided release': buckets[1],
        'No. of Top 25% DBH trees with 0-sided release': buckets[0]
    }
    return pd.DataFrame([out])


tbl_vs_initial_tfa2 = table_final_vs_initial(
    df_initial=df,
    df_final=df_tfa2,
    metric='pre_DBH', vol_col='pre_stem_vol',
    status_col='status', thin_col='thin_decision',
    strategy='Thin from above-1 vs Initial'
).round(3).set_index('Strategy')

tbl_vs_after_first_tfa2 = table_final_vs_after_first(
    df_after_first=df_best3,
    df_final=df_tfa2,
    metric='pre_DBH', vol_col='pre_stem_vol',
    status_col='status', thin_col='thin_decision',
    strategy='Thin from above-1 vs After 3-row'
).round(3).set_index('Strategy')

release_tfa1 = top25_release_table(
    df_after_first=df_best3,
    df_final=df_tfa2,                 
    treatment='Thin from above-1',
    dbh_col='pre_DBH',
    row_col='Row', tree_col='Tree',
    status_col='status', thin_col='thin_decision',
    keep_val='Keep', thin_val='Thin',
    neighbors_k=5, row_scale=1.0, tree_scale=1.0
).round(3).set_index('Treatment')

# display(tbl_vs_initial_tfa2)
# display(tbl_vs_after_first_tfa2)
# display(release_tfa1)

# plot_thinning_map(df_tfa2, row_col='Row', x_col='Tree', status_col='status',
#                   title='Thin from Above-1 - Spatial Map')

### Thin from Above-2

In [9]:
def _euclid2d(a_rows, a_trees, b_rows, b_trees, row_scale=1.0, tree_scale=1.0):
    dr = (a_rows[:, None] - b_rows[None, :]) * row_scale
    dt = (a_trees[:, None] - b_trees[None, :]) * tree_scale
    return np.sqrt(dr * dr + dt * dt)

def thin_from_above_phase2_anchor_immediate5(
    df_best3,
    *,
    dbh_col='pre_DBH',
    row_col='Row',
    tree_col='Tree',
    status_col='status',
    thin_col='thin_decision',
    keep_val='Keep',
    thin_val='Thin',
    # anchors & window
    top_pct_anchors=0.10,     
    k_neighbors=5,            
    # budget
    stand_target_removed_frac=1/3,
    target_rounding='round',  
    reserve_for_phaseB=0.0,   
    row_scale=1.0,
    tree_scale=1.0,
    verbose=False
):
    """
    THin from Above-2 logic:
      • Build, for each anchor, the list of its 5 nearest residual neighbors (Alive&Keep after 3-row),
        INCLUDING other anchors; exclude self. This window is FIXED for all phases.
      • CUTTING may thin only those neighbors in the window that are non-anchors (anchors are skipped).
      • COUNTING uses the SAME window; sides released = # of those 5 that end up Thin.

    """
    d0 = df_best3.copy()

    baseline_mask = d0[status_col].eq('Alive') & d0[thin_col].eq(keep_val)
    baseline_idx  = d0.index[baseline_mask]
    n_base = len(baseline_idx)
    if n_base == 0:
        return d0, {'error': 'Empty residual baseline after 3-row'}

    baseline = d0.loc[baseline_idx]

    if target_rounding == 'floor':
        target_total = int(np.floor(n_base * stand_target_removed_frac))
    elif target_rounding == 'ceil':
        target_total = int(np.ceil(n_base * stand_target_removed_frac))
    else:
        target_total = int(np.round(n_base * stand_target_removed_frac))
    target_total = max(0, min(target_total, n_base))

    reserve = int(np.floor(float(reserve_for_phaseB) * target_total))
    phaseA_cap = max(0, target_total - reserve)

    n_anchors = max(1, int(np.ceil(n_base * float(top_pct_anchors))))
    anchors_idx = baseline.nlargest(n_anchors, dbh_col).index
    anchors_df  = baseline.loc[anchors_idx].sort_values(dbh_col, ascending=False)
    anchors_set = set(anchors_idx)

    B_rows  = baseline[row_col].to_numpy(float)
    B_trees = baseline[tree_col].to_numpy(float)
    B_ids   = np.array(list(baseline_idx))
    id2pos  = {B_ids[i]: i for i in range(len(B_ids))}

    immediate5 = {}           
    cuttable5  = {}           

    for a in anchors_df.index:
        ai = id2pos[a]
        D = _euclid2d(
            np.array([B_rows[ai]]), np.array([B_trees[ai]]),
            B_rows, B_trees, row_scale=row_scale, tree_scale=tree_scale
        ).ravel()

        D[ai] = np.inf  # exclude self
        k = min(int(k_neighbors), len(B_ids)-1)
        order = np.argpartition(D, k-1)[:k]
        order = order[np.argsort(D[order])]  
        neigh_ids = B_ids[order]
        immediate5[a] = neigh_ids

        cuttable5[a]  = [nid for nid in neigh_ids if nid not in anchors_set]

    all_cuttable = pd.Index(np.unique([nid for lst in cuttable5.values() for nid in lst]))
    max_possible_from_windows = int(len(all_cuttable))

    d1 = d0.copy()

    picksA = []
    budgetA = phaseA_cap
    for a in anchors_df.index:
        if budgetA <= 0:
            break
        for nid in cuttable5[a]:
            if budgetA <= 0:
                break
            if d1.at[nid, thin_col] != keep_val:
                continue
            d1.at[nid, thin_col] = thin_val
            picksA.append(nid)
            budgetA -= 1

    removedA = int(len(picksA))
    remaining = target_total - removedA
    remaining = max(0, remaining)

    picksB = []
    phaseB_triggered = remaining > 0
    if phaseB_triggered:
        for a in anchors_df.index:
            if remaining <= 0:
                break
            
            window_ids = immediate5[a]
            cut_mask = d1.loc[window_ids, thin_col].eq(thin_val)
        
            for nid in cuttable5[a]:
                if remaining <= 0:
                    break
                if d1.at[nid, thin_col] == keep_val:
                    d1.at[nid, thin_col] = thin_val
                    picksB.append(nid)
                    remaining -= 1

    # ---------- Counting----------
    def _count_release_buckets(df_view):
        # returns counts for k = 0..5 (k = number of Thin in the window)
        buckets = {k: 0 for k in range(0, k_neighbors+1)}
        for a in anchors_df.index:
            window_ids = immediate5[a]
            if len(window_ids) == 0:
                buckets[0] += 1
                continue
            k_cut = int(df_view.loc[window_ids, thin_col].eq(thin_val).sum())
            k_cut = int(max(0, min(k_cut, k_neighbors)))
            buckets[k_cut] += 1
        return buckets

    buckets_after = _count_release_buckets(d1)

    # ---------- Diagnostics ----------
    info = {
        'baseline_count': int(n_base),
        'target_removed_total': int(target_total),
        'removed_phaseA': removedA,
        'removed_phaseB': int(len(pd.Index(picksB).unique())),
        'removed_total': int(removedA + len(pd.Index(picksB).unique())),
        'unused_budget_after_B': int(remaining),
        'phaseB_triggered': bool(phaseB_triggered),
        'anchors_count': int(len(anchors_idx)),
        'capacity_immediate5_nonanchors_unique': int(max_possible_from_windows),
        'release_buckets_after': {int(k): int(v) for k, v in buckets_after.items()},
    }

    if verbose:
        print(f"[TFA2-immediate5] baseline={n_base} target={target_total} "
              f"A={removedA} B={len(picksB)} unused={remaining} "
              f"anchors={len(anchors_idx)} capacity={max_possible_from_windows} "
              f"release_after={buckets_after}")

    return d1, info


### Thin from Above-2 - Tiered

In [10]:
def _euclid2d(a_rows, a_trees, b_rows, b_trees, row_scale=1.0, tree_scale=1.0):
    dr = (a_rows[:, None] - b_rows[None, :]) * row_scale
    dt = (a_trees[:, None] - b_trees[None, :]) * tree_scale
    return np.sqrt(dr * dr + dt * dt)

def _build_immediate5_windows(baseline_df, anchor_ids, *, row_col, tree_col,
                              k_neighbors=5, row_scale=1.0, tree_scale=1.0):
    B_ids   = baseline_df.index.to_numpy()
    B_rows  = baseline_df[row_col].to_numpy(float)
    B_trees = baseline_df[tree_col].to_numpy(float)
    id2pos  = {B_ids[i]: i for i in range(len(B_ids))}
    out = {}
    for a in anchor_ids:
        ai = id2pos[a]
        D = _euclid2d(
            np.array([B_rows[ai]]), np.array([B_trees[ai]]),
            B_rows, B_trees, row_scale=row_scale, tree_scale=tree_scale
        ).ravel()
        D[ai] = np.inf
        k = min(int(k_neighbors), len(B_ids) - 1)
        order = np.argpartition(D, k - 1)[:k]
        order = order[np.argsort(D[order])]
        out[a] = B_ids[order]
    return out

def thin_from_above2_tiered_immediate5(
    df_best3,
    *,
    # columns
    dbh_col='pre_DBH',
    vol_col='pre_stem_vol',
    row_col='Row',
    tree_col='Tree',
    status_col='status',
    thin_col='thin_decision',
    keep_val='Keep',
    thin_val='Thin',
    # windows
    k_neighbors=5,
    row_scale=1.0, tree_scale=1.0,
    # tiers
    tier1_frac=0.10,              # top 10% = Tier-1
    tier2_frac=0.15,              # next 15% = Tier-2
    # budget
    stand_target_removed_frac=1/3,
    target_rounding='round',      
    reserve_for_tier2_ratio=0.30,
    # verbosity
    verbose=False
):
    """
    Tiered-release:

    Phase 0: Build tiers on the post-3-row residual.
    Phase 1: Process Tier-1 anchors (largest→smallest). In each anchor's immediate-5, CUT any neighbor
             that is not Tier-1 (i.e., Tier-2 anchors and non-anchors). Tier-1 are never cut.
    Phase 2: Process Tier-2 anchors that SURVIVED Phase 1 (largest→smallest). In their immediate-5, CUT
             neighbors that are neither Tier-1 nor surviving Tier-2. (So Tier-1 and surviving Tier-2 are protected.)
    """
    d0 = df_best3.copy()

    # Baseline
    baseline_mask = d0[status_col].eq('Alive') & d0[thin_col].eq(keep_val)
    baseline_idx  = d0.index[baseline_mask]
    if len(baseline_idx) == 0:
        return d0, {'error': 'Empty baseline after 3-row'}

    baseline = d0.loc[baseline_idx]
    n_base   = len(baseline_idx)

    # Budget
    if target_rounding == 'floor':
        target_total = int(np.floor(n_base * stand_target_removed_frac))
    elif target_rounding == 'ceil':
        target_total = int(np.ceil(n_base * stand_target_removed_frac))
    else:
        target_total = int(np.round(n_base * stand_target_removed_frac))
    target_total = max(0, min(target_total, n_base))
    phase1_cap   = int(max(0, target_total * (1.0 - float(reserve_for_tier2_ratio))))

    # Tiers (by DBH)
    baseline_sorted = baseline.sort_values(dbh_col, ascending=False)
    n_t1      = max(1, int(np.ceil(n_base * float(tier1_frac))))
    tier1_ids = baseline_sorted.index[:n_t1]

    n_t2_add   = max(0, int(np.ceil(n_base * float(tier2_frac))))
    tier2_pool = baseline_sorted.index.difference(tier1_ids)
    tier2_ids  = tier2_pool[:n_t2_add]

    tier1_set = set(tier1_ids)
    tier2_set = set(tier2_ids)

    # Immediate-5 windows
    windows_t1 = _build_immediate5_windows(baseline, tier1_ids, row_col=row_col, tree_col=tree_col,
                                           k_neighbors=k_neighbors, row_scale=row_scale, tree_scale=tree_scale)
    windows_t2 = _build_immediate5_windows(baseline, tier2_ids, row_col=row_col, tree_col=tree_col,
                                           k_neighbors=k_neighbors, row_scale=row_scale, tree_scale=tree_scale)

    # Phase 1 (Tier-1 release)
    d1 = d0.copy()
    picks1 = []
    for a in tier1_ids:
        if len(picks1) >= phase1_cap:
            break
        for nid in windows_t1[a]:
            if len(picks1) >= phase1_cap:
                break
            if nid in tier1_set:
                continue  # never cut Tier-1
            if d1.at[nid, thin_col] == keep_val:
                d1.at[nid, thin_col] = thin_val
                picks1.append(nid)

    removed1 = int(len(picks1))
    remaining_budget = max(0, target_total - removed1)

    # Surviving Tier-2 after Phase 1
    tier2_survivors = [t for t in tier2_ids if d1.at[t, thin_col] == keep_val]
    t2_survivor_set = set(tier2_survivors)

    # Phase 2 (Tier-2 release)
    picks2 = []
    if remaining_budget > 0 and len(tier2_survivors) > 0:
        t2_sorted = baseline.loc[tier2_survivors].sort_values(dbh_col, ascending=False).index
        for a in t2_sorted:
            if len(picks2) >= remaining_budget:
                break
            for nid in windows_t2[a]:
                if len(picks2) >= remaining_budget:
                    break
                if (nid in tier1_set) or (nid in t2_survivor_set):
                    continue  # protect Tier-1 and surviving Tier-2
                if d1.at[nid, thin_col] == keep_val:
                    d1.at[nid, thin_col] = thin_val
                    picks2.append(nid)

    removed2      = int(len(picks2))
    total_removed = removed1 + removed2
    unused_budget = max(0, target_total - total_removed)

    # Diagnostics payload
    info = {
        'baseline_count': int(n_base),
        'target_removed_total': int(target_total),
        'removed_phase1': removed1,
        'removed_phase2': removed2,
        'removed_total': total_removed,
        'unused_budget': int(unused_budget),
        'tier1': {'count': int(len(tier1_ids)), 'ids': list(tier1_ids)},
        'tier2': {'count': int(len(tier2_ids)), 'ids': list(tier2_ids),
                  'survivors_after_p1': int(len(tier2_survivors))}
    }
    if verbose:
        print(f"[TFA2-Tiered] base={n_base} target={target_total} "
              f"P1={removed1} P2={removed2} total={total_removed} unused={unused_budget} "
              f"T1={len(tier1_ids)} T2={len(tier2_ids)}")
    return d1, info


### Dropdown

In [21]:
pd.set_option('display.max_columns', None)   
pd.set_option('display.width', 0)            
pd.set_option('display.max_colwidth', None) 

def anchor_release_table_immediate5(
    df_after_first, df_final, *,
    treatment: str,
    top_pct_anchors: float = 0.10,
    neighbors_k: int = 5,
    dbh_col='pre_DBH',
    row_col='Row',
    tree_col='Tree',
    status_col='status',
    thin_col='thin_decision',
    keep_val='Keep',
    thin_val='Thin',
    row_scale: float = 1.0,
    tree_scale: float = 1.0
) -> pd.DataFrame:

    base_mask = df_after_first[status_col].eq('Alive') & df_after_first[thin_col].eq(keep_val)
    base_idx  = df_after_first.index[base_mask]
    if len(base_idx) == 0:
        cols = ['Treatment','Initial tree quantity','Post-thin tree quantity',
                'pre-thin mean DBH','post-thin mean DBH',
                'Anchors (count)','Anchors cut off (count)'] + \
               [f'No. of anchors with {k}-sided release' for k in [5,4,3,2,1,0]]
        return pd.DataFrame([dict(zip(cols,[treatment,0,0,np.nan,np.nan,0,0,0,0,0,0,0]))])

    d0 = df_after_first.loc[base_idx]
    d1 = df_final.loc[base_idx]

    initial_qty = len(base_idx)
    post_keep_mask = d1[thin_col].eq(keep_val)
    post_qty = int(post_keep_mask.sum())
    pre_mean_dbh  = float(d0[dbh_col].astype(float).mean())
    post_mean_dbh = float(d0.loc[post_keep_mask, dbh_col].astype(float).mean()) if post_qty>0 else np.nan

    n_base = len(base_idx)
    n_anchors = max(1, int(np.ceil(n_base * float(top_pct_anchors))))
    anchors_idx = d0.nlargest(n_anchors, dbh_col).index

    
    anchors_cut_count = int(d1.loc[anchors_idx, thin_col].eq(thin_val).sum())
    anchors_alive_idx = anchors_idx.difference(d1.index[d1[thin_col].eq(thin_val)])

    # Immediate-5 window per anchor
    B_rows  = d0[row_col].to_numpy(float)
    B_trees = d0[tree_col].to_numpy(float)
    B_ids   = np.array(list(base_idx))
    id2pos  = {B_ids[i]: i for i in range(len(B_ids))}

    buckets = {k: 0 for k in range(0, neighbors_k+1)}

    for a in anchors_alive_idx:
        ai = id2pos[a]
        D = _euclid2d(np.array([B_rows[ai]]), np.array([B_trees[ai]]), B_rows, B_trees,
                      row_scale=row_scale, tree_scale=tree_scale).ravel()
        D[ai] = np.inf
        k = min(neighbors_k, len(B_ids)-1)
        order = np.argpartition(D, k-1)[:k]
        order = order[np.argsort(D[order])]
        window_ids = B_ids[order]
        k_cut = int(d1.loc[window_ids, thin_col].eq(thin_val).sum())
        k_cut = max(0, min(k_cut, neighbors_k))
        buckets[k_cut] += 1

    out = {
        'Treatment': treatment,
        'Initial tree quantity': initial_qty,
        'Post-thin tree quantity': post_qty,
        'pre-thin mean DBH': pre_mean_dbh,
        'post-thin mean DBH': post_mean_dbh,
        'Anchors (count)': int(len(anchors_idx)),
        'Anchors cut off (count)': anchors_cut_count,
        'No. of anchors with 5-sided release': buckets[5],
        'No. of anchors with 4-sided release': buckets[4],
        'No. of anchors with 3-sided release': buckets[3],
        'No. of anchors with 2-sided release': buckets[2],
        'No. of anchors with 1-sided release': buckets[1],
        'No. of anchors with 0-sided release': buckets[0]
    }
    return pd.DataFrame([out])



def _stand_metrics_relative(
    base_df, final_df, base_mask, strategy,
    metric='pre_DBH', vol_col='pre_stem_vol',
    status_col='status', thin_col='thin_decision',
    keep_val='Keep', thin_val='Thin',
    include_thinned_stats: bool = False,   # NEW: off by default
):
    base_idx = base_df.index[base_mask]
    base_idx = base_idx.intersection(final_df.index)

    if len(base_idx) == 0:
        return {
            'Strategy': strategy,
            'Trees removed(%)': 0.0, 'Volume removed(%)': 0.0,
            'Trees kept': 0, 'Trees removed': 0,
            'Q1 cut (count)': 0,
            'Q4 remaining (count)': 0, 'Q4 cut (count)': 0,
            'Removed volume-Q4(%)': 0.0, 'Removed volume-Q4': 0.0,
            'Change-Median DBH': np.nan, 'Change-Mean DBH': np.nan,
            'Volume removed': 0.0, 'Post-thinning total volume': 0.0,
            'Q1 removal ratio': np.nan, 'Q4 retention ratio': np.nan,
            'Pre-thinning Median DBH': np.nan, 'Pre-thinning Mean DBH': np.nan,
            'Post-thinning Median DBH': np.nan, 'Post-thinning Mean DBH': np.nan,
            # New fields (empty)
            'Thinned Median DBH': np.nan, 'Thinned Mean DBH': np.nan,
        }

    pre  = base_df.loc[base_idx]    # baseline cohort (e.g., after 3-row)
    post = final_df.loc[base_idx]   # same cohort after secondary thinning

    keep_mask = post[thin_col].eq(keep_val)
    cut_mask  = post[thin_col].eq(thin_val)

    n_base = len(base_idx)
    n_kept = int(keep_mask.sum())
    n_cut  = int(cut_mask.sum())

    base_vol  = float(pre[vol_col].sum())
    vol_cut   = float(pre.loc[cut_mask, vol_col].sum())
    vol_post  = float(pre.loc[keep_mask, vol_col].sum())

    trees_removed_pct = 100.0 * (n_cut / n_base) if n_base else 0.0
    vol_removed_pct   = 100.0 * (vol_cut / base_vol) if base_vol > 0 else 0.0

    x = pre[metric].astype(float)
    q1_thr = float(x.quantile(0.25))
    q3_thr = float(x.quantile(0.75))
    q1_mask = x <= q1_thr
    q4_mask = x >= q3_thr

    q1_cut = int((q1_mask & cut_mask).sum())
    q4_keep = int((q4_mask & keep_mask).sum())
    q4_cut  = int((q4_mask & cut_mask).sum())

    vol_cut_q4     = float(pre.loc[q4_mask & cut_mask, vol_col].sum())
    vol_cut_q4_pct = 100.0 * (vol_cut_q4 / vol_cut) if vol_cut > 0 else 0.0

    pre_med  = float(x.median()); pre_mean  = float(x.mean())
    x_post   = pre.loc[keep_mask, metric].astype(float)
    post_med = float(x_post.median()) if len(x_post) > 0 else np.nan
    post_mean= float(x_post.mean())   if len(x_post) > 0 else np.nan

    change_med  = post_med - pre_med  if pd.notnull(post_med)  else np.nan
    change_mean = post_mean - pre_mean if pd.notnull(post_mean) else np.nan

    base_q1 = int(q1_mask.sum()); base_q4 = int(q4_mask.sum())
    q1_removal_ratio = (q1_cut / base_q1) if base_q1 > 0 else np.nan
    q4_retention_ratio = (q4_keep / base_q4) if base_q4 > 0 else np.nan

    out = {
        'Strategy': strategy,
        'Trees removed(%)': trees_removed_pct,
        'Volume removed(%)': vol_removed_pct,
        'Trees kept': n_kept,
        'Trees removed': n_cut,
        'Q1 cut (count)': q1_cut,
        'Q4 remaining (count)': q4_keep,
        'Q4 cut (count)': q4_cut,
        'Removed volume-Q4(%)': vol_cut_q4_pct,
        'Removed volume-Q4': vol_cut_q4,
        'Change-Median DBH': change_med,
        'Change-Mean DBH': change_mean,
        'Volume removed': vol_cut,
        'Post-thinning total volume': vol_post,
        'Q1 removal ratio': q1_removal_ratio,
        'Q4 retention ratio': q4_retention_ratio,
        'Pre-thinning Median DBH': pre_med,
        'Pre-thinning Mean DBH': pre_mean,
        'Post-thinning Median DBH': post_med,
        'Post-thinning Mean DBH': post_mean,
    }


    if include_thinned_stats:
        x_cut = pre.loc[cut_mask, metric].astype(float)
        out['Thinned Median DBH'] = float(x_cut.median()) if len(x_cut) > 0 else np.nan
        out['Thinned Mean DBH']   = float(x_cut.mean())   if len(x_cut) > 0 else np.nan
    else:
        out['Thinned Median DBH'] = np.nan
        out['Thinned Mean DBH']   = np.nan

    return out


def table_final_vs_initial(
    df_initial, df_final, *,
    metric='pre_DBH', vol_col='pre_stem_vol',
    status_col='status', thin_col='thin_decision',
    keep_val='Keep', thin_val='Thin',
    strategy='Final vs Initial (Alive baseline)'
):
  
    base_mask = df_initial[status_col].eq('Alive')
    rep = _stand_metrics_relative(
        df_initial, df_final, base_mask, strategy,
        metric, vol_col, status_col, thin_col, keep_val, thin_val,
        include_thinned_stats=False     # keep this table unchanged
    )
    return pd.DataFrame([rep])


def table_final_vs_after_first(
    df_after_first, df_final, *,
    metric='pre_DBH', vol_col='pre_stem_vol',
    status_col='status', thin_col='thin_decision',
    keep_val='Keep', thin_val='Thin',
    strategy='Final vs After 1st Thinning (Alive&Keep baseline)'
):
=
    base_mask = df_after_first[status_col].eq('Alive') & df_after_first[thin_col].eq(keep_val)
    rep = _stand_metrics_relative(
        df_after_first, df_final, base_mask, strategy,
        metric, vol_col, status_col, thin_col, keep_val, thin_val,
        include_thinned_stats=True      
    )
    return pd.DataFrame([rep])



display(HTML("<style>.jp-OutputArea-output { text-align:center; }</style>"))

_df_initial = globals().get('df_initial', globals().get('df'))
if _df_initial is None:
    raise NameError("Please define your initial stand as `df_initial` (or `df`).")
if 'df_best3' not in globals():
    raise NameError("Please define `df_best3` as the stand after the 3-row thinning.")


display(HTML("<style>.jp-OutputArea-output { text-align:center; }</style>"))

_df_initial = globals().get('df_initial', globals().get('df'))
if _df_initial is None:
    raise NameError("Please define your initial stand as `df_initial` (or `df`).")
if 'df_best3' not in globals():
    raise NameError("Please define `df_best3` as the stand after the 3-row thinning.")

# Global anchor fraction
ANCHOR_FRAC = 0.25

def _run_and_show(strategy_key):
    if strategy_key == 'tfb':
        df_out, _ = thin_from_below_adjacent_simple(
            df_best3,
            fraction=1/3,
            metric='pre_DBH',
            row_col='Row',
            status_col='status'
        )
        label = 'Thin from below'
        show_release = False

    elif strategy_key == 'tfa1':
        df_out, _ = thin_from_above_neighbors(
            df_best3,
            removal_fraction=1/3,
            metric='pre_DBH',
            row_col='Row',
            x_col='Tree',
            status_col='status',
            thin_col='thin_decision',
            keep_val='Keep',
            thin_val='Thin',
            anchor_fraction=0.10,
            min_anchors=1,
            radius=None,
            combine='max'
        )
        label = 'Thin from above-1'
        show_release = True

    elif strategy_key == 'tfa2':
        df_out, info = thin_from_above_phase2_anchor_immediate5(
            df_best3,
            dbh_col='pre_DBH',
            row_col='Row', tree_col='Tree',
            status_col='status', thin_col='thin_decision',
            keep_val='Keep', thin_val='Thin',
            top_pct_anchors=ANCHOR_FRAC,  # keep in sync with table
            k_neighbors=5,
            stand_target_removed_frac=1/3,
            target_rounding='round',
            reserve_for_phaseB=0.10,
            row_scale=1.0, tree_scale=1.0,
            verbose=False
        )
        label = 'Thin from above-2'
        show_release = True

        # diagnostics
        print(f"PhaseBTriggered={info['phaseB_triggered']}, "
              f"RemovedA={info['removed_phaseA']}, RemovedB={info['removed_phaseB']}, "
              f"UnusedBudget={info['unused_budget_after_B']}, "
              f"CapacityWithinWindows={info['capacity_immediate5_nonanchors_unique']}")

    elif strategy_key == 'tfa2_tiered':
        
        TIER1_FRAC = 0.25
        TIER2_FRAC = 0.0
        df_out, info = thin_from_above2_tiered_immediate5(
            df_best3,
            dbh_col='pre_DBH', vol_col='pre_stem_vol',
            row_col='Row', tree_col='Tree',
            status_col='status', thin_col='thin_decision',
            keep_val='Keep', thin_val='Thin',
            k_neighbors=5, row_scale=1.0, tree_scale=1.0,
            tier1_frac=TIER1_FRAC, tier2_frac=TIER2_FRAC,
            stand_target_removed_frac=1/3, target_rounding='round',
            reserve_for_tier2_ratio=0,
            verbose=False
        )
        label = 'Thin from above-2-Tiered'
        show_release = True

        # diagnostics
        print(f"P1={info['removed_phase1']} P2={info['removed_phase2']} "
              f"TotalRemoved={info['removed_total']} UnusedBudget={info['unused_budget']} "
              f"T1={info['tier1']['count']} T2={info['tier2']['count']}")

    else:
        raise ValueError("Unknown strategy key")

   
    tbl_vs_initial = table_final_vs_initial(
        df_initial=_df_initial,
        df_final=df_out,
        metric='pre_DBH', vol_col='pre_stem_vol',
        status_col='status', thin_col='thin_decision',
        strategy=f'{label} vs Initial'
    ).round(3).set_index('Strategy')

    tbl_vs_after_first = table_final_vs_after_first(
        df_after_first=df_best3,
        df_final=df_out,
        metric='pre_DBH', vol_col='pre_stem_vol',
        status_col='status', thin_col='thin_decision',
        strategy=f'{label} vs After 3-row'
    ).round(3).set_index('Strategy')

    display(tbl_vs_initial)
    display(tbl_vs_after_first)

    if show_release and strategy_key in ('tfb','tfa1','tfa2', 'tfa2_tiered'):
        release_tbl = anchor_release_table_immediate5(
            df_after_first=df_best3,
            df_final=df_out,
            treatment=label,
            top_pct_anchors=ANCHOR_FRAC,   # use the global fraction for consistency
            neighbors_k=5,
            dbh_col='pre_DBH', row_col='Row', tree_col='Tree',
            status_col='status', thin_col='thin_decision',
            keep_val='Keep', thin_val='Thin',
            row_scale=1.0, tree_scale=1.0
        ).round(3).set_index('Treatment')
        display(release_tbl)


    plot_thinning_map(df_out, row_col='Row', x_col='Tree', status_col='status', title=label)
    plt.show()


dd = widgets.Dropdown(
    options=[('Thin from below','tfb'),
             ('Thin from above-1','tfa1'),
             ('Thin from above-2','tfa2'),
             # ('Thin from above-2-Tiered','tfa2_tiered')
            ],
    value='tfb',
    description='Strategy:',
    style={'description_width':'110px'},
    layout=widgets.Layout(width='420px')
)
out = widgets.Output()

def _on_change(change):
    if change['name']=='value' and change['type']=='change':
        with out:
            clear_output(wait=True)
            _run_and_show(change['new'])

dd.observe(_on_change, names='value')
display(dd, out)

with out:
    clear_output(wait=True)
    _run_and_show(dd.value)


Dropdown(description='Strategy:', layout=Layout(width='420px'), options=(('Thin from below', 'tfb'), ('Thin fr…

Output()